# Bosonic qubits in dynamiqs — a hands-on probing notebook

**What this notebook is.** A guided tour of the three main families of *exotic* bosonic
qubits — dissipative cats, Kerr cats, multi-legged cats and GKP grid states — built up by
simulation rather than by derivation. Every section poses a question, runs a numerical
experiment, and states what the result *means* physically.

**Who it is for.** Someone comfortable with Python and with basic quantum mechanics, but
who has not spent years in the bosonic-codes literature. No result below is assumed; each
one is something you can watch happen on screen.

**The single question the whole notebook circles around:**

> A harmonic oscillator has an infinite-dimensional Hilbert space. If we hide one qubit
> inside it *redundantly*, can we arrange the physics so that the noise which actually
> occurs in the lab damages only one kind of logical information, and essentially never
> the other?

The answer is yes, and the asymmetry is the entire point.

---

## 0. Conventions and vocabulary

### Operators

The annihilation operator $\mathbf{a}$ and the quadratures:

$$\mathbf{q} = \frac{\mathbf{a}+\mathbf{a}^\dagger}{\sqrt 2}, \qquad
\mathbf{p} = -i\,\frac{\mathbf{a}-\mathbf{a}^\dagger}{\sqrt 2}, \qquad [\mathbf{q},\mathbf{p}]=i .$$

A coherent state $|\alpha\rangle$ sits at the point $(\mathrm{Re}\,\alpha, \mathrm{Im}\,\alpha)$
in the phase-space plots produced by `dq.plot.wigner`. Its mean photon number is $|\alpha|^2$.

### Open-system dynamics

Everything here is a Lindblad master equation,

$$\dot{\boldsymbol\rho} = -i[\mathbf{H},\boldsymbol\rho] \;+\; \sum_k \mathcal{D}[\mathbf{L}_k](\boldsymbol\rho),
\qquad
\mathcal{D}[\mathbf{L}](\boldsymbol\rho) = \mathbf{L}\boldsymbol\rho \mathbf{L}^\dagger
- \tfrac12 \mathbf{L}^\dagger \mathbf{L}\boldsymbol\rho - \tfrac12 \boldsymbol\rho \mathbf{L}^\dagger\mathbf{L}.$$

In `dynamiqs` this is `dq.mesolve(H, L, rho0, tsave)` where `L` is the **list of jump
operators $\mathbf{L}_k$ themselves**, not rates. A rate $\kappa$ therefore appears as
`jnp.sqrt(kappa) * op`. Forgetting the square root silently sets your rate to 1.

### The one concept to internalise: **dark states**

If a state satisfies $\mathbf{L}_k|\psi\rangle = 0$ for *every* jump operator, and
$\mathbf{H}=0$, then $\dot{\boldsymbol\rho}=0$ exactly. The state is frozen. Every
stabilisation scheme below is built by *designing* $\mathbf{L}$ so that the states you want
to protect are its dark states. That is the whole trick:

> **Reservoir engineering = choosing the kernel of $\mathbf{L}$ to be your code space.**

### Two kinds of logical error (memorise this)

For a cat qubit encoded with $|0_L\rangle=|{+}\alpha\rangle$, $|1_L\rangle=|{-}\alpha\rangle$:

| error | what moves | logical Pauli | protected? |
|---|---|---|---|
| **bit-flip** | population between the $+\alpha$ and $-\alpha$ lobes | $X$ | **yes**, exponentially in $|\alpha|^2$ |
| **phase-flip** | coherence between the lobes / photon-number parity | $Z$ | no, gets *worse* $\propto|\alpha|^2$ |

Useful observables, and what each one diagnoses:

| observable | formula | diagnoses |
|---|---|---|
| lobe populations | $\langle \pm\alpha \vert \boldsymbol\rho \vert \pm\alpha\rangle$ | bit-flips |
| parity | $\langle \boldsymbol\Pi\rangle$, $\boldsymbol\Pi=(-1)^{\mathbf{a}^\dagger\mathbf{a}}$ | phase-flips |
| branch coherence | $\left\vert \langle \alpha \vert \boldsymbol\rho \vert {-}\alpha\rangle \right\vert$ | phase-flips |
| mean photon number | $\langle \mathbf{a}^\dagger\mathbf{a}\rangle$ | leakage / energy |
| purity | $\mathrm{Tr}(\boldsymbol\rho^2)$ | total decoherence |

### A practical warning about this notebook

Cells share global variables (`a`, `K`, `eta`, `eps`, `na`, ...) and several cells **redefine
them**. For example `a` becomes a two-mode operator in the buffer sections and a single-mode
one again later; `K` is set to `2.5` in the Kerr section and to `0.25` in the flow-field
section. **Run the notebook top to bottom.** If you re-run a cell out of order and get
nonsense, suspect a stale global before suspecting physics.

---
# Part 1 — The dissipative cat

## 1.1 Two-photon dissipation, the adiabatic (reduced) model

The simplest possible stabilised cat: no Hamiltonian at all, one engineered jump operator

$$\mathbf{L} = \sqrt{\kappa_2}\,\big(\mathbf{a}^2 - \alpha_0^2\big).$$

**Why this works.** Since $\mathbf{a}|\alpha\rangle = \alpha|\alpha\rangle$, we have
$\mathbf{L}|\pm\alpha_0\rangle = \sqrt{\kappa_2}(\alpha_0^2-\alpha_0^2)|\pm\alpha_0\rangle = 0$.
Both coherent states — and hence *every superposition of them* — are exact dark states. The
dark space is two-dimensional:

$$\mathcal{C} = \mathrm{span}\{|{+}\alpha_0\rangle, |{-}\alpha_0\rangle\}
= \mathrm{span}\{|\mathcal{C}^+_{\alpha_0}\rangle, |\mathcal{C}^-_{\alpha_0}\rangle\},
\qquad
|\mathcal{C}^\pm_{\alpha_0}\rangle \propto |\alpha_0\rangle \pm |{-}\alpha_0\rangle .$$

**Term-by-term:**

- $\mathbf{a}^2$ — removes photons **two at a time**. Crucially this *conserves photon-number
  parity*, so it cannot mix the even cat $|\mathcal{C}^+\rangle$ with the odd cat
  $|\mathcal{C}^-\rangle$. This is exactly why the scheme protects the logical $X$ basis.
- $-\alpha_0^2$ — the constant offset that *chooses where the dark states live*. Change it
  and you move the lobes; drop it and $\mathbf{L}=\mathbf{a}^2$ just drains the oscillator to
  vacuum.
- $\sqrt{\kappa_2}$ — sets only the **speed** of convergence, not the destination.

Below, $\alpha_0^2 = 4$ so $\alpha_0 = 2$, and we start from **vacuum**. Vacuum has even
parity, and $\mathbf{a}^2$ preserves parity, so the state is funnelled into the *even* cat.

**What to watch in the gif:** a single blob at the origin stretches along the real axis, splits
into two lobes at $x=\pm 2$, and interference fringes appear between them along $p$. Those
fringes are the signature of a genuine quantum superposition — a classical mixture of
$|{\pm}\alpha_0\rangle$ would show the two lobes and *nothing* in between.

In [ ]:
import jax.numpy as jnp
import matplotlib.pyplot as plt
import dynamiqs as dq
import io
from IPython.display import Image, display

k2 = 1 

a = dq.destroy(32)

H = jnp.zeros((32,32))

L = [jnp.sqrt(k2) * (a@a - 4 * dq.eye(32))]

psi0 = dq.fock(32,0)

tsave = jnp.linspace(0, 2.0, 100)

result = dq.mesolve(H,L,psi0,tsave)

gif = dq.plot.wigner_gif(result.states, fps = 30, xmax = 6, ymax = 3)

display(gif)

|██████████| 100.0% ◆ elapsed 195.86ms ◆ remaining 0.00ms 
 73%|███████▎  | 73/100 [00:05<00:01, 16.19it/s]

### Reading the result

- **Convergence is monotonic and irreversible.** Unlike Hamiltonian evolution, nothing
  oscillates or comes back. The cat is an *attractor*, not an orbit.
- **Where the negativity comes from.** The fringes are Wigner-negative. They were not present
  in vacuum and no unitary was applied — the dissipator itself generated the non-classicality,
  which is the counter-intuitive heart of reservoir engineering.
- **Convergence time.** The rate onto the manifold scales roughly as $\kappa_2|\alpha_0|^2$,
  so with $\kappa_2=1,\ |\alpha_0|^2=4$ the state is essentially converged well before
  $t=2$.

## 1.2 Changing $\kappa_2$, and starting *inside* the manifold

Two independent changes here, worth separating mentally:

1. **$\kappa_2 = 2$ instead of 1.** Pure speed. The steady state is *identical*; only the
   approach is twice as fast. $\kappa_2$ is the knob for "how hard am I pinning the state",
   not "where am I pinning it".
2. **A different initial state**: $|\psi_0\rangle \propto |2i\rangle + |{-}2i\rangle$, i.e. a
   cat on the **imaginary** axis, while the dissipator's dark states are on the **real** axis
   ($\alpha_0^2=+4 \Rightarrow \alpha_0=\pm2$).

Point 2 is the interesting one. Note that $|{\pm}2i\rangle$ satisfies
$\mathbf{a}^2|{\pm}2i\rangle = (2i)^2|{\pm}2i\rangle = -4|{\pm}2i\rangle \ne +4|{\pm}2i\rangle$,
so this state is **as far from dark as it gets**: it sits at the anti-fixed-point of the
dissipator.

**What to watch:** the state does *not* simply rotate to the real axis. It gets destroyed and
re-formed — the $y$-axis lobes decay while the $x$-axis lobes grow. The dissipator has no
rotation in it at all (no Hamiltonian!), so it cannot "move" a lobe; it can only kill
amplitude where the state is bright and grow it where the dark states are.

In [ ]:
k2 = 2

a = dq.destroy(32)

H = jnp.zeros((32,32))

L = [jnp.sqrt(k2) * (a@a - 4 * dq.eye(32))]

psi0 = (dq.coherent(32,2j) + dq.coherent(32,-2j)).unit() #dq.fock(32,0)

tsave = jnp.linspace(0, 1, 400)

result = dq.mesolve(H,L,psi0,tsave)

gif = dq.plot.wigner_gif(result.states, fps = 30, xmax = 6, ymax = 3)

display(gif)

### Reading the result

This is the clearest demonstration that **dissipative stabilisation does not conserve the
state you hand it** — it only conserves states already inside its kernel. Hand it something
orthogonal to the code space and the information is simply erased, not rotated. Compare with
the Kerr cat later, where a Hamiltonian *does* rotate states around.

## 1.3 Where does $\mathbf{L}=\sqrt{\kappa_2}(\mathbf{a}^2-\alpha_0^2)$ come from? The buffer

Nothing in nature removes photons two at a time. The engineered dissipator is *derived* from
a two-mode circuit: a long-lived **memory** $\mathbf{a}$ coupled to a heavily damped
**buffer** $\mathbf{b}$.

$$\mathbf{H} = g_2\left(\mathbf{a}^{\dagger 2}\mathbf{b} + \mathbf{a}^2\mathbf{b}^\dagger\right)
\;+\; \epsilon_2\left(\mathbf{b}^\dagger + \mathbf{b}\right),
\qquad \mathbf{L} = \sqrt{\kappa_b}\,\mathbf{b}.$$

**Term-by-term — this is the most important table in the notebook:**

| term | what it physically does | what it does to information |
|---|---|---|
| $g_2(\mathbf{a}^{\dagger2}\mathbf{b} + \mathbf{a}^2\mathbf{b}^\dagger)$ | coherent 2-photon $\leftrightarrow$ 1-photon exchange (three-wave mixing, needs a Josephson element) | **moves** entropy from memory into buffer; reversible on its own |
| $\epsilon_2(\mathbf{b}^\dagger+\mathbf{b})$ | linear drive on the buffer | **chooses** the target: sets $\alpha_0^2$ |
| $\sqrt{\kappa_b}\,\mathbf{b}$ | ordinary single-photon loss of the lossy mode | **destroys** entropy irreversibly — the only true sink in the circuit |

Remove the loss and nothing is stabilised: the $g_2$ term alone just sloshes excitations back
and forth between memory and buffer forever. Remove $g_2$ and the loss never reaches the
memory. **Stabilisation requires the relay of both.**

### Adiabatic elimination

If the buffer relaxes much faster than it is driven, we can solve for its steady state,
$\langle \mathbf{b}\rangle_{ss} = -2i(g_2\mathbf{a}^2+\epsilon_2)/\kappa_b$, substitute back,
and recover exactly the one-mode dissipator of §1.1 with

$$\boxed{\;\kappa_2 = \frac{4 g_2^2}{\kappa_b}, \qquad \alpha_0^2 = -\frac{\epsilon_2}{g_2}\;}$$

**Mind the minus sign.** To reproduce the $\mathbf{a}^2 - 4$ of §1.1 you need
$\epsilon_2 = -g_2\alpha_0^2$ (here $= -40$). With the *opposite* sign you get
$\alpha_0^2 = -4$, i.e. $\alpha_0 = 2i$ — a perfectly good cat, but rotated onto the imaginary
axis, which is a very easy way to think your simulation is broken when it isn't.

### The validity condition

$$\kappa_b \;\gg\; 2 g_2 |\alpha_0| \qquad \text{(bad-cavity / fast-buffer limit)}$$

The cell below uses $g_2=10 \Rightarrow \kappa_b = 400$, giving
$2g_2|\alpha_0| = 40 \ll 400$. Comfortably adiabatic.

**What to watch:** the memory (first gif) reproduces §1.1. The buffer (second gif) is the more
instructive one — it stays a small blob very near the origin, barely populated
($|\langle\mathbf{b}\rangle| \sim 2|\epsilon_2|/\kappa_b \approx 0.2$). *That* is what "the
buffer is adiabatically slaved" looks like: it never accumulates anything, it just passes
excitations straight through to the environment.

In [ ]:
k2, alpha2 = 1.0, 4.0
na, nb = 32, 8
tsave = jnp.linspace(0, 1.0, 100)

# full two-mode
g2 = 10
kb = 4 * g2**2 / k2        # = 100
e2 = -g2 * alpha2          # = -20  -> alpha^2 = -e2/g2 = +4

a = dq.destroy(na) & dq.eye(nb)
b = dq.eye(na) & dq.destroy(nb)
H = g2 * (a.dag() @ a.dag() @ b + a @ a @ b.dag()) + e2 * (b.dag() + b)
res_full = dq.mesolve(H, [jnp.sqrt(kb) * b], dq.fock(na, 0) & dq.fock(nb, 0), tsave)

rho_a = dq.ptrace(res_full.states, 0)
rho_b = dq.ptrace(res_full.states, 1)

gif1 = dq.plot.wigner_gif(rho_a, fps = 30, xmax = 6, ymax = 3)


gif2 = dq.plot.wigner_gif(rho_b, fps = 30, xmax = 6, ymax = 3)
display(gif1)
display(gif2)




### Reading the result

If you ever see the buffer gif show a *large* or *oscillating* blob, adiabatic elimination has
broken down and the reduced model no longer describes your system. The buffer's emptiness is a
free, sensitive diagnostic — use it.

## 1.4 How good is adiabatic elimination? A quantitative check

We now compare the reduced model against the full two-mode simulation, at fixed
$\kappa_2 = 4g_2^2/\kappa_b$.

**The key insight this cell is designed to expose:** for a *fixed* effective rate $\kappa_2$,
the pair $(g_2,\kappa_b)$ is not unique. Since $\kappa_b = 4g_2^2/\kappa_2$, matched pairs
scale **together**:

| $g_2$ | $\kappa_b = 4g_2^2/\kappa_2$ | $\kappa_2$ | $2g_2\alpha_0 / \kappa_b$ | adiabatic? |
|---|---|---|---|---|
| 0.5 | 1 | 1 | $2.0$ | **no** — buffer is slower than the coupling |
| 2.0 | 16 | 1 | $0.5$ | marginal |
| 10.0 | 400 | 1 | $0.1$ | yes |

So a weak coupling needs only a weakly lossy buffer, and a strong coupling demands a
*very* lossy buffer. What matters is never $g_2$ alone — it is always the **ratio**.

> ⚠️ **Note on this cell's plot labels.** The legend strings (`"kb=1"`, `"kb=16"`,
> `"k2=16 mismatched"`) are leftovers and do not match what `run_full` computes — the function
> always sets `kb = 4*g2**2/1`, so **all three curves have $\kappa_2=1$**; they differ only in
> how adiabatic they are. The physics in the plot is right; the labels are stale. Reading them
> as $g_2 = 0.5,\ 2.0,\ 10.0$ gives the correct interpretation.

In [ ]:
import jax.numpy as jnp
import matplotlib.pyplot as plt
import dynamiqs as dq

na, nb = 32, 8
alpha2 = 4.0
alpha = jnp.sqrt(alpha2)
tsave = jnp.linspace(0, 10.0, 100)

def even_cat(n, alpha):
    c = dq.coherent(n, alpha) + dq.coherent(n, -alpha)
    return dq.unit(c)

cat_target = even_cat(na, alpha)

def fidelity_series(states, target):
    try:
        return dq.fidelity(states, target)
    except Exception:
        # manual fallback: F = <cat| rho |cat> for a batch of density matrices
        psi = target.to_numpy().reshape(-1)   # (na,)
        rhos = states.to_numpy()               # (T, na, na)
        return jnp.real(jnp.einsum('i,tij,j->t', psi.conj(), rhos, psi))

def run_adiabatic(k2):
    a = dq.destroy(na)
    L = [jnp.sqrt(k2) * (a @ a - alpha2 * dq.eye(na))]
    return dq.mesolve(dq.zeros(na), L, dq.fock(na, 0), tsave)

def run_full(g2):
    kb = 4 * g2**2 / 1
    e2 = -g2 * alpha2                      # correct sign -> alpha^2 = +alpha2
    a = dq.destroy(na) & dq.eye(nb)
    b = dq.eye(na) & dq.destroy(nb)
    H = g2 * (a.dag() @ a.dag() @ b + a @ a @ b.dag()) + e2 * (b.dag() + b)
    res = dq.mesolve(H, [jnp.sqrt(kb) * b], dq.fock(na, 0) & dq.fock(nb, 0), tsave)
    return dq.ptrace(res.states, 0)

# --- baseline: effect of k2 alone (your first two scripts) ---
res_ad1 = run_adiabatic(1.0)
res_ad2 = run_adiabatic(2.0)

# --- effect of g2: three (g2, kb) pairs, all targeting kappa2 = 1 ---
rho_weak   = run_full(g2=0.5)    
rho_strong = run_full(g2=10.0)
rho_ring   = run_full(g2=2.0)

F_ad1    = fidelity_series(res_ad1.states, cat_target)
F_ad2    = fidelity_series(res_ad2.states, cat_target)
F_weak   = fidelity_series(rho_weak,   cat_target)
F_strong = fidelity_series(rho_strong, cat_target)
F_ring   = fidelity_series(rho_ring,   cat_target)

plt.figure(figsize=(7,5))
plt.plot(tsave, F_ad1,    label="adiabatic, k2=1")
plt.plot(tsave, F_ad2,    label="adiabatic, k2=2")
plt.plot(tsave, F_weak,   '--', label="full, g2=0.5, kb=1   (k2=1, matched)")
plt.plot(tsave, F_strong, '--', label="full, g2=2.0, kb=16  (k2=1, matched)")
plt.plot(tsave, F_ring,   ':',  label="full, g2=2.0, kb=1   (k2=16, mismatched)")
plt.xlabel("time"); plt.ylabel("fidelity to even cat |α|²=4")
plt.legend(fontsize=8); plt.show()

### Reading the result

- The two solid curves (reduced model, $\kappa_2=1$ vs $2$) show **rate only** — same
  destination, different speed.
- The $g_2=10$ dashed curve should lie essentially **on top of** the $\kappa_2=1$ reduced
  curve. That agreement *is* the validation of adiabatic elimination.
- The $g_2=0.5$ curve should visibly **lag or ring**: with $\kappa_b=1 < 2g_2\alpha_0=2$, the
  buffer is not fast enough, excitations slosh back into the memory, and the reduced
  description fails.
- Fidelity saturates slightly **below 1** even in the best case. That residual is real: at
  finite $\alpha_0$ the target `even_cat` is not exactly the steady state, and the
  finite-$\kappa_b$ correction adds a small deterministic error. Perfect agreement would
  actually be suspicious.

---
# Part 2 — The Kerr cat: stabilisation *without* dissipation

An entirely different philosophy. Instead of engineering a dissipator whose kernel is the code
space, engineer a **Hamiltonian whose ground-state manifold** is the code space:

$$\mathbf{H} = -K\,\mathbf{a}^{\dagger2}\mathbf{a}^2 \;+\; \epsilon_2\left(\mathbf{a}^{\dagger2}+\mathbf{a}^2\right).$$

**Term-by-term:**

- $-K\mathbf{a}^{\dagger2}\mathbf{a}^2$ — the **Kerr nonlinearity**: a photon-number-dependent
  frequency shift. Alone, it makes a coherent state shear and fold, generating negativity and
  fractional revivals. It supplies the "double-well" curvature.
- $\epsilon_2(\mathbf{a}^{\dagger2}+\mathbf{a}^2)$ — a **two-photon (squeezing) drive**. It
  breaks rotational symmetry and picks out an axis.

Together they form a phase-space double well with minima at $\pm\alpha_0$, where

$$\alpha_0^2 = \frac{\epsilon_2}{K}.$$

The two cat states $|\mathcal{C}^\pm_{\alpha_0}\rangle$ are the (nearly) degenerate **ground
doublet**, split only by an exponentially small tunnelling gap
$\Delta \sim e^{-c|\alpha_0|^2}$ — that exponentially small splitting *is* the bit-flip
protection.

### The crucial difference from Part 1

| | dissipative cat | Kerr cat |
|---|---|---|
| code space is... | the **kernel of $\mathbf{L}$** | the **ground manifold of $\mathbf{H}$** |
| mechanism | entropy removal | energy gap |
| arbitrary state converges to it? | **yes**, it is an attractor | **no** — unitary evolution has no attractors |
| protects against | leaving the manifold | leaving the manifold |

That third row matters enormously in practice. A pure Kerr Hamiltonian started in vacuum will
**not** settle into a cat; it will circulate forever, because energy is conserved. To actually
*prepare* the state you must add a relaxation channel. Below, that is single-photon loss
$\mathbf{L}=\sqrt{\kappa_1}\mathbf{a}$, which damps population down into the ground doublet.

Here $\epsilon_2=10,\ K=2.5 \Rightarrow \alpha_0^2 = 4$, matching the cats of Part 1.

In [ ]:
alpha2 = 4
e2 = 10
K = e2 / alpha2

a = dq.destroy(32)
H = -K * a.dag() @ a.dag() @ a @ a + e2 * (a.dag() @ a.dag() + a @ a)

psi0 = dq.fock(32, 0)
tsave = jnp.linspace(0, 1.0, 100)

kappa1 = 0.1
L = [jnp.sqrt(kappa1) * a]
result = dq.mesolve(H, L, psi0, tsave)

gif = dq.plot.wigner_gif(result.states, fps=30, xmax=6, ymax=3)
display(gif)

### Reading the result

- With $\kappa_1 = 0.1$ you should see vacuum spread and begin to organise into two lobes —
  but the dynamics look **qualitatively different** from §1.1: more swirling, more structure,
  because the Kerr Hamiltonian is actively rotating things while $\kappa_1$ slowly damps them.
- Try setting `kappa1 = 0` and re-running (use `dq.sesolve(H, psi0, tsave)` — `mesolve` with
  an empty `L` list errors out). You should see **no convergence at all**: collapses and
  revivals forever. That single comparison is the cleanest demonstration that the Kerr
  Hamiltonian confines but does not *attract*.
- Note the irony worth savouring: $\kappa_1$ is the enemy in Part 1 (it causes phase-flips) but
  is the *preparation mechanism* here. Same operator, opposite role, depending on what else is
  switched on.

### 2.1 Starting from a deliberately contaminated state

Now we hand the Kerr Hamiltonian a state that is mostly a good cat plus a 10% "impurity"
along the imaginary axis:

$$|\psi_0\rangle \propto |2\rangle + |{-}2\rangle + 0.1\,|2i\rangle.$$

The impurity sits **outside** the ground-state wells, i.e. at higher energy.

**What to watch:** the two main lobes stay put (they are near-eigenstates), while the small
contaminant orbits, smears, and is gradually damped away by $\kappa_1$. This is the Kerr cat's
version of "error correction": the energy gap plus weak relaxation returns population to the
protected manifold.

In [ ]:
alpha2 = 4
e2 = 10
K = e2 / alpha2

a = dq.destroy(32)
H = -K * a.dag() @ a.dag() @ a @ a + e2 * (a.dag() @ a.dag() + a @ a)

psi0 = dq.coherent(32, 2) + dq.coherent(32, -2) + 0.1 * dq.coherent(32, 2j)
psi0 = psi0.unit()
tsave = jnp.linspace(0, 1.0, 100)

kappa1 = 0.1
L = [jnp.sqrt(kappa1) * a]
result = dq.mesolve(H, L, psi0, tsave)

gif = dq.plot.wigner_gif(result.states, fps=30, xmax=6, ymax=3)
display(gif)

### Reading the result

The contaminant does not vanish instantly — it **rings down** at a rate set by $\kappa_1$ and
by its energy above the doublet. This is the qualitative difference from dissipative
stabilisation, where out-of-manifold components are killed directly by $\kappa_2$ at a rate
you control by design rather than inherit from the hardware.

---
# Part 3 — Phase-space flow fields: *seeing* the difference

The most illuminating single picture in this notebook. Take the mean-field (semiclassical)
equation of motion for $\alpha = \langle\mathbf{a}\rangle$ and plot it as a vector field.

**Two-photon dissipation:**
$$\frac{d\alpha}{dt} = -\kappa_2\,\bar\alpha\,\left(\alpha^2 - \alpha_0^2\right)$$

**Kerr alone:**
$$\frac{d\alpha}{dt} = 2i\,\bar\alpha\left(K\alpha^2 - \epsilon_2\right)$$

**Kerr + single-photon loss:**
$$\frac{d\alpha}{dt} = 2i\,\bar\alpha\left(K\alpha^2 - \epsilon_2\right) - \frac{\kappa_1}{2}\alpha$$

**The thing to notice is the factor of $i$.** In the dissipative case the drift coefficient is
**real** — the flow points *along* the line toward $\pm\alpha_0$, so those points are genuine
**sinks**. In the pure-Kerr case it is **purely imaginary** — the flow is everywhere
*perpendicular* to the displacement, giving **closed orbits** around $\pm\alpha_0$, never into
them. Adding $-\tfrac{\kappa_1}{2}\alpha$ reintroduces a real, inward component and restores a
sink.

This is the same conclusion as §2, now visible as geometry rather than inferred from a movie.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_flow(drift, title, lim=4, n=25, fixed_pts=None):
    x = np.linspace(-lim, lim, n); y = np.linspace(-lim, lim, n)
    X, Y = np.meshgrid(x, y)
    A = X + 1j*Y
    dA = drift(A)
    U, V = np.real(dA), np.imag(dA)
    plt.figure(figsize=(5,5))
    plt.streamplot(X, Y, U, V, color=np.hypot(U,V), cmap='viridis', density=1.3)
    if fixed_pts is not None:
        plt.scatter(np.real(fixed_pts), np.imag(fixed_pts), c='red', marker='x', s=80)
    plt.xlabel('Re(α)'); plt.ylabel('Im(α)'); plt.title(title)
    plt.gca().set_aspect('equal'); plt.show()

alpha2 = 4.0; alpha0 = np.sqrt(alpha2)
k2 = 1.0; K = 0.25; eps2 = K*alpha2; kappa1 = 2

plot_flow(lambda a: -k2*np.conj(a)*(a**2-alpha2),
          "two-photon dissipation: real sink at ±α0", fixed_pts=[alpha0,-alpha0])

plot_flow(lambda a: 2j*np.conj(a)*(K*a**2-eps2),
          "Kerr alone: closed orbits, no sink", fixed_pts=[alpha0,-alpha0])

plot_flow(lambda a: 2j*np.conj(a)*(K*a**2-eps2) - 0.5*kappa1*a,
          "Kerr + κ1: now a real sink", fixed_pts=[alpha0,-alpha0])

### Reading the three plots

1. **Two-photon dissipation** — streamlines converge cleanly onto the two red crosses. There is
   a separatrix along the imaginary axis dividing the two basins; crossing it is a bit-flip.
2. **Kerr alone** — streamlines *circle* the crosses. No arrow points inward. Nothing converges.
   This is the picture of "confined but not attracted", and it retro-explains why the earlier
   Kerr simulation moved around strangely instead of settling.
3. **Kerr + $\kappa_1$** — spirals. The orbital motion from Kerr is still there, now with an
   inward drift, so trajectories corkscrew into the wells.

### What the flow field *cannot* show you

This is worth stating explicitly because it is a genuine conceptual limit. The mean-field drift
describes a single point $\alpha$ in phase space. **Bit-flips** are exactly a question about
that point — which basin does it fall into, how hard is it to cross the separatrix — so the
flow field is the right tool.

**Phase-flips are invisible here.** The states $|\alpha_0\rangle$,
$|\mathcal{C}^+\rangle$ and $|\mathcal{C}^-\rangle$ all put probability mass in the *same*
places; what distinguishes them is the relative phase between the lobes, which lives in the
*interference fringes between* them and is not a coordinate the classical drift has access to.
There is therefore no arrow you could draw that represents "restoring the phase" — not because
it is weak, but because **no such restoring force exists**. Phase-flip protection has to be
diagnosed with a different observable entirely, which is what Part 4 does.

---
# Part 4 — Phase-flips: parity decay under single-photon loss

## The exact algebraic fact

Apply the annihilation operator to an even cat:

$$\mathbf{a}\,|\mathcal{C}^+_\alpha\rangle \;\propto\; \mathbf{a}\big(|\alpha\rangle + |{-}\alpha\rangle\big)
= \alpha|\alpha\rangle - \alpha|{-}\alpha\rangle \;\propto\; |\mathcal{C}^-_\alpha\rangle .$$

This is **exact** — no large-$\alpha$ approximation. A single photon loss maps the even cat
*precisely* onto the odd cat.

In logical terms, with $\{|{\pm}\alpha\rangle\}$ as the $Z$ basis, the parity states
$\{|\mathcal{C}^\pm\rangle\}$ are the $X$ basis. Photon loss flips the $X$ states, i.e. it
applies a **logical $Z$**: a **phase-flip**, not a bit-flip. Under stabilisation, single-photon
loss is therefore, to excellent approximation, **pure dephasing**.

Averaged over many random jump times (which is what `mesolve` does), a Poissonian stream of
$Z$ kicks becomes a smooth $T_2$-type decay of $\langle\boldsymbol\Pi\rangle$.

## The scaling, and why the bias exists

The jump rate is $\kappa_1\langle \mathbf{a}^\dagger\mathbf{a}\rangle = \kappa_1|\alpha|^2$, so

$$\Gamma_{\text{phase-flip}} \;\propto\; \kappa_1 |\alpha|^2 \qquad\text{(grows linearly)}$$

while, as Part 6 will measure,

$$\Gamma_{\text{bit-flip}} \;\propto\; e^{-c|\alpha|^2} \qquad\text{(falls exponentially)}.$$

**Increasing the cat size trades phase-flips for bit-flips**, and it is a spectacularly good
trade: linear cost, exponential gain. This is the "noise bias" that the whole cat-qubit
programme is built on — you then spend a classical repetition code correcting only the
phase-flips.

The cell below starts from a *perfect* even cat (so parity starts at $+1$) and watches it
decay, for three cat sizes.

In [ ]:
def parity_op(n):
    return dq.asqarray(jnp.diag((-1.0)**jnp.arange(n)))

def even_cat(n, alpha):
    return dq.unit(dq.coherent(n, alpha) + dq.coherent(n, -alpha))

Pi = parity_op(32)
kappa1 = 0.05
tsave = jnp.linspace(0, 20, 200)

for a2 in [2, 4, 6]:
    eps2 = K*a2; alpha0 = jnp.sqrt(a2)
    H = -K*a.dag()@a.dag()@a@a + eps2*(a.dag()@a.dag()+a@a)
    res = dq.mesolve(H, [jnp.sqrt(kappa1)*a], even_cat(32, alpha0), tsave, exp_ops=[Pi])
    plt.plot(tsave, res.expects[0].real, label=f"|α|²={a2}")
plt.xlabel("time"); plt.ylabel("⟨parity⟩"); plt.legend()

### Reading the result

- All three curves decay from $+1$ toward $0$. Parity $\to 0$ means an equal mixture of even
  and odd — total loss of $X$-basis information.
- **Bigger cats decay faster.** $|\alpha|^2=6$ should decay roughly three times faster than
  $|\alpha|^2=2$. Confirm the spacing is roughly *linear* in $|\alpha|^2$ — compare with the
  log-scale plot in Part 6, which is *exponential* in the same parameter. Seeing the two
  scalings side by side is the point.
- Parity does **not** revive. Dephasing is irreversible in the ensemble-averaged description.
  (In a single quantum trajectory each jump is a discrete, in-principle-trackable event; the
  smooth decay is what survives after averaging over unknown jump times.)

> ℹ️ This cell uses the global `K` — which was reset to `0.25` by the flow-field cell in
> Part 3, not the `2.5` of Part 2. So $\epsilon_2 = K|\alpha|^2$ here corresponds to a
> different (weaker) Kerr than the movies above. Harmless for the parity scaling being
> demonstrated, but a good example of the stale-global hazard flagged in §0.

---
# Part 5 — Switching stabilisation on and off

A three-phase protocol, run twice, designed to isolate what stabilisation actually buys you.

| phase | $t$ | sequence **A** | sequence **B** |
|---|---|---|---|
| 1 | 0–1 | stabilise ($\kappa_2$) | stabilise ($\kappa_2$) — identical, reused |
| 2 | 1–2 | **$\kappa_2$ OFF**, $\kappa_1$ ON | **$\kappa_2$ stays ON**, $\kappa_1$ ON |
| 3 | 2–3 | stabilise again | $\kappa_1$ off, stabilise |

Both sequences experience the *same* single-photon loss during phase 2. The only difference is
whether the confinement was present while the damage was being done. This is the numerical
analogue of "is my error-correction on or off during the noise?".

**Predictions worth writing down before running:**

- **Sequence A, phase 2:** with no confinement, $\kappa_1$ simply drains the oscillator toward
  vacuum. Expect $\langle n\rangle$ to **fall**, both lobe populations to collapse, and the
  state to leave the cat manifold entirely ($F_+ + F_- $ drops well below 1).
- **Sequence B, phase 2:** confinement holds $\langle n\rangle$ near $|\alpha_0|^2$. The loss
  cannot change the energy, so it expresses itself almost purely as **parity decay**, with
  $F_+ + F_-$ staying near 1. Population moves *between* $|\mathcal{C}^+\rangle$ and
  $|\mathcal{C}^-\rangle$ rather than *out* of their span.
- **Phase 3, both:** re-stabilisation restores the manifold ($\langle n\rangle$ and
  $F_++F_-$ recover) — but **purity does not fully recover**, and parity does not return to
  $+1$. Stabilisation restores *which manifold you are in*; it cannot restore *which state
  within it* you were.

That last point is the honest statement of what autonomous stabilisation does and does not do:
it corrects leakage, not logical errors.

In [ ]:
# ---------------------------------------------------------------- helpers ---
def to_array(x):
    """Get a raw (T, n, n) or (n, n) jax array out of a dynamiqs QArray."""
    try:
        return x.to_jax()
    except AttributeError:
        return jnp.asarray(x.to_numpy())

def run_segment(H, L, rho0, T, npts):
    tsave = jnp.linspace(0.0, T, npts)
    res = dq.mesolve(H, L, rho0, tsave)
    return res, tsave

# ------------------------------------------------------------- parameters ---
na = 32
alpha2 = 4.0
alpha0 = jnp.sqrt(alpha2)

k2 = 5.0          # strong two-photon stabilisation rate
k1 = k2 / 3        # single-photon loss, half that strength

a  = dq.destroy(na)
H0 = dq.zeros(na)

L_stab = [jnp.sqrt(k2) * (a @ a - alpha2 * dq.eye(na))]   # stabilisation only
L_loss = [jnp.sqrt(k1) * a]                               # loss only, no stabilisation
L_both = [jnp.sqrt(k2) * (a @ a - alpha2 * dq.eye(na)),   # stabilisation + loss together
          jnp.sqrt(k1) * a]

T_phase   = 1.0
n_per_seg = 100

# --------------------------------------------------------- sequence A run ---
# stabilise -> switch OFF stabilisation, apply loss alone -> switch stabilisation back on
psi0 = dq.fock(na, 0)

resA1, t1 = run_segment(H0, L_stab, psi0,            T_phase, n_per_seg)
resA2, t2 = run_segment(H0, L_loss, resA1.states[-1], T_phase, n_per_seg)
resA3, t3 = run_segment(H0, L_stab, resA2.states[-1], T_phase, n_per_seg)

# --------------------------------------------------------- sequence B run ---
# stabilise -> KEEP stabilisation on, add the same loss on top -> drop the loss again
resB1, _ = resA1, t1                                   # identical first phase, reuse
resB2, _ = run_segment(H0, L_both, resA1.states[-1], T_phase, n_per_seg)
resB3, _ = run_segment(H0, L_stab, resB2.states[-1], T_phase, n_per_seg)

# ---------------------------------------------------- stitch each sequence ---
def stitch(res_list, t_list):
    """Concatenate states/time across phases, dropping the duplicated boundary point."""
    arrs = [to_array(res_list[0].states)]
    times = [t_list[0]]
    offset = t_list[0][-1]
    for res, t in zip(res_list[1:], t_list[1:]):
        arrs.append(to_array(res.states)[1:])     # drop first point (== previous last point)
        times.append(offset + t[1:])
        offset = offset + t[-1]
    return jnp.concatenate(arrs, axis=0), jnp.concatenate(times, axis=0)

statesA, tA = stitch([resA1, resA2, resA3], [t1, t2, t3])
statesB, tB = stitch([resB1, resB2, resB3], [t1, t2, t3])

# ------------------------------------------------------------------ gifs ---
gifA = dq.plot.wigner_gif(dq.asqarray(statesA), fps=30, xmax=6, ymax=6)

gifB = dq.plot.wigner_gif(dq.asqarray(statesB), fps=30, xmax=6, ymax=6)
display(gifA)
display(gifB)

# ------------------------------------------------------- observables setup ---
def cat_ket(n, alpha, sign):
    c = dq.coherent(n, alpha) + sign * dq.coherent(n, -alpha)
    return to_array(dq.unit(c))          # shape (n, 1)

ket_plus  = cat_ket(na, alpha0, +1)
ket_minus = cat_ket(na, alpha0, -1)
proj_plus  = ket_plus  @ ket_plus.conj().T
proj_minus = ket_minus @ ket_minus.conj().T

n_op      = jnp.diag(jnp.arange(na).astype(jnp.float64))
parity_op = jnp.diag((-1.0) ** jnp.arange(na))

def expect_series(op, rho_arr):
    # Tr(op @ rho) for each time slice, rho_arr: (T, n, n)
    return jnp.real(jnp.einsum('ij,tji->t', op, rho_arr))

def purity_series(rho_arr):
    return jnp.real(jnp.einsum('tij,tji->t', rho_arr, rho_arr))

def compute_all(states):
    return dict(
        F_plus  = expect_series(proj_plus,  states),
        F_minus = expect_series(proj_minus, states),
        n_mean  = expect_series(n_op,       states),
        parity  = expect_series(parity_op,  states),
        purity  = purity_series(states),
    )

obsA = compute_all(statesA)
obsB = compute_all(statesB)

# ------------------------------------------------------------------ plots ---
fig, axes = plt.subplots(4, 1, figsize=(8, 11), sharex=True)

for (obs, t, label, ls) in [(obsA, tA, "A (stab OFF mid-phase)", '-'),
                            (obsB, tB, "B (stab stays ON)",      '--')]:
    axes[0].plot(t, obs['F_plus'],  ls, label=f"{label}: |C+>")
    axes[0].plot(t, obs['F_minus'], ls, label=f"{label}: |C->")
    axes[0].plot(t, obs['F_plus'] + obs['F_minus'], ls, lw=2,
                 label=f"{label}: sum")
    axes[1].plot(t, obs['n_mean'], ls, label=label)
    axes[2].plot(t, obs['parity'], ls, label=label)
    axes[3].plot(t, obs['purity'], ls, label=label)

for ax in axes:
    ax.axvline(1.0, color='gray', lw=0.8, ls=':')
    ax.axvline(2.0, color='gray', lw=0.8, ls=':')
    ax.legend(fontsize=7)

axes[0].set_ylabel("cat-manifold population")
axes[1].set_ylabel("⟨n⟩")
axes[2].set_ylabel("⟨parity⟩")
axes[3].set_ylabel("purity Tr(ρ²)")
axes[3].set_xlabel("time")
fig.suptitle("Phase 1: stabilise | Phase 2: perturb | Phase 3: re-stabilise")
plt.tight_layout()
plt.show()

### Reading the four panels

| panel | A (stab off) | B (stab on) | meaning |
|---|---|---|---|
| $F_+, F_-$, and their **sum** | sum dips hard in phase 2 | sum stays $\approx 1$ | leakage out of the code space |
| $\langle n\rangle$ | falls toward 0 | roughly flat | confinement fixes the energy |
| $\langle\boldsymbol\Pi\rangle$ | decays | decays | phase-flips happen **either way** |
| purity | drops, partial recovery | drops, partial recovery | decoherence is not undone |

The headline: **the sum $F_+ + F_-$ is the leakage diagnostic; parity is the logical-error
diagnostic.** Stabilisation fixes the first and is powerless against the second. If you take
one plot away from this notebook, make it this one.

---
# Part 6 — Bit-flips: are they *really* exponentially suppressed?

## 6.1 The idealised model has *zero* bit-flip rate

Consider pure two-photon dissipation with $\kappa_1 = 0$. Since
$\mathbf{L}|{\pm}\alpha_0\rangle = 0$ **exactly**, and $\mathbf{H}=0$, every state in the code
space satisfies $\dot{\boldsymbol\rho} = 0$ *identically*. Not "slowly evolving" — frozen.

So in the ideal model the bit-flip rate is not small, it is **exactly zero**. Nothing
accumulates, ever. The cell below verifies this numerically over a long run: start in
$|{+}\alpha_0\rangle$ and check the populations do not budge.

This is a genuinely useful habit — before measuring a small effect, verify that it is *absent*
when it should be. If `max drift` comes out at $10^{-6}$ rather than $10^{-1}$, you know your
subsequent measurement is signal, not solver error.

In [ ]:
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
import dynamiqs as dq

na = 40
a = dq.destroy(na)
H0 = dq.zeros(na)

def cat_projectors(n, alpha0):
    ket_p = dq.unit(dq.coherent(n, alpha0))
    ket_m = dq.unit(dq.coherent(n, -alpha0))
    return ket_p, ket_m

def run(alpha2, k2, k1, T, npts):
    alpha0 = jnp.sqrt(alpha2)
    L = [jnp.sqrt(k2) * (a @ a - alpha2 * dq.eye(na)),
         jnp.sqrt(k1) * a]
    psi0 = dq.coherent(na, alpha0)
    tsave = jnp.linspace(0.0, T, npts)
    res = dq.mesolve(H0, L, psi0, tsave)
    ket_p = dq.unit(dq.coherent(na, alpha0))
    ket_m = dq.unit(dq.coherent(na, -alpha0))
    F_plus  = jnp.real(dq.expect(ket_p @ ket_p.dag(), res.states))
    F_minus = jnp.real(dq.expect(ket_m @ ket_m.dag(), res.states))
    return tsave, F_plus, F_minus

# ---------------------------------------------------------------------------
# Part 1: idealized model, k1 = 0 -> should be an EXACT dark state, forever
# ---------------------------------------------------------------------------
k2 = 4.0
alpha2_demo = 4.0
T_long = 40.0

t, Fp, Fm = run(alpha2_demo, k2=k2, k1=0.0, T=T_long, npts=200)

plt.figure(figsize=(6,4))
plt.plot(t, Fp, label="F(+α0)")
plt.plot(t, Fm, label="F(-α0)")
plt.ylim(-0.05, 1.05)
plt.xlabel("time"); plt.ylabel("population")
plt.title(f"κ1=0, |α0|²={alpha2_demo}: pure κ2 dissipation — should stay flat forever")
plt.legend(); plt.show()

print("max drift in F(+α0) over the whole run:", float(jnp.max(jnp.abs(Fp - Fp[0]))))
print("max leakage into F(-α0):", float(jnp.max(Fm)))

### 6.2 Now add $\kappa_1$ and measure the rate

Real bit-flips come from imperfections that push the state *off* the dark subspace — here,
single-photon loss. Once slightly off the manifold, there is a small probability of relaxing
into the **wrong** well. In the phase-space picture of Part 3 this is a noise-activated escape
over the separatrix, the quantum analogue of Kramers' problem.

We start in $|{+}\alpha_0\rangle$ and track the population imbalance

$$\mathcal{I}(t) = F_+(t) - F_-(t) \;\sim\; e^{-2\Gamma_{\text{bf}} t},$$

fitting $\Gamma_{\text{bf}}$ from the slope of $\log\mathcal{I}$. Unlike the ideal case, this
**does** accumulate linearly in time like any rate process — it is just astronomically slow for
large cats.

The sweep over $|\alpha_0|^2$ is the payoff: on a log axis, $\Gamma_{\text{bf}}$ should fall
close to a **straight line**, i.e. exponential suppression.

In [ ]:
# ---------------------------------------------------------------------------
# Part 2: add a small κ1, extract the bit-flip rate vs |α0|²
# ---------------------------------------------------------------------------
k1 = 0.05          # kept << k2 so we stay deep in the stabilized regime
T_bf = 40.0
npts = 50

alpha2_list = [1.0, 1.5, 2.0, 2.5, 3.0, 3.5]
rates = []

plt.figure(figsize=(6,4))
for a2 in alpha2_list:
    t, Fp, Fm = run(a2, k2=k2, k1=k1, T=T_bf, npts=npts)
    diff = Fp - Fm                      # -> decays like exp(-2*Gamma_bf*t)
    diff = jnp.clip(diff, 1e-6, None)   # guard against numerical negatives near 0
    logdiff = jnp.log(diff)

    # linear fit on the (still linear-looking) early portion, before it flattens near 0
    mask = diff > 0.05
    slope = jnp.polyfit(t[mask], logdiff[mask], 1)[0]
    gamma_bf = -slope / 2.0
    rates.append(float(gamma_bf))

    plt.plot(t, diff, label=f"|α0|²={a2}")

plt.xlabel("time"); plt.ylabel("F(+α0) − F(-α0)")
plt.legend(fontsize=8); plt.title("population imbalance decaying via bit-flip")
plt.show()

rates = np.array(rates)
plt.figure(figsize=(6,4))
plt.semilogy(alpha2_list, rates, 'o-')
plt.xlabel("|α0|²"); plt.ylabel("extracted bit-flip rate Γ_bf (log scale)")
plt.title(f"bit-flip rate vs cat size, κ1={k1}, κ2={k2}")
plt.show()

print(list(zip(alpha2_list, rates)))

### Reading the result

- **First plot:** the imbalance decays from 1 toward 0 — the state forgetting which lobe it
  started in. Bigger cats decay dramatically more slowly.
- **Second plot (log scale):** roughly linear decrease $\Rightarrow \Gamma_{\text{bf}} \sim
  e^{-c|\alpha_0|^2}$. **Compare directly with Part 4's parity decay**, which grew *linearly*
  in $|\alpha_0|^2$. Same knob, opposite scalings — the noise bias, measured.
- **Why the modest $|\alpha_0|^2$ range (1 to 3.5)?** Above that, $\Gamma_{\text{bf}}$ becomes
  too small to resolve within the simulated window, and the fit degrades into noise. That
  numerical wall is itself the physics: "exponentially suppressed" means you quickly reach
  rates you cannot measure. Real experiments hit the same wall and have to run for seconds.
- If points at the high end look non-monotonic, increase `T_bf` or raise `k1` uniformly across
  the sweep — don't read a wiggle there as physics.

---
# Part 7 — Watching a bit-flip happen in phase space

The population plots above are abstract. Here we make the leak visible: start in
$|{+}\alpha_0\rangle$, use a deliberately **small cat** ($|\alpha_0|^2=2$) and a deliberately
**large** $\kappa_1$ so that the process is fast enough to see.

Two views:

1. **The gif** — the right lobe fades while the left lobe grows.
2. **A kymograph** (space–time diagram) — a 1D cut of the Wigner function through both lobes,
   stacked vertically as time advances. Two bright vertical stripes at $x=\pm\alpha_0$; watch
   the weight transfer from one to the other.

> ⚠️ **Check the cut axis.** The cell takes `cut = W[:, :, y0_idx]`. Whether that is the
> $y=0$ cut or an $x=$const cut depends on the axis ordering `dq.wigner` returns for batched
> states. Verify once with an asymmetric test state (e.g. a single `dq.coherent(n, 2)`, whose
> $y=0$ cut must show a single peak at $x=+2$); if the stripes look wrong, swap to
> `W[:, y0_idx, :]`.

In [ ]:
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
import dynamiqs as dq

na = 40
a = dq.destroy(na)
H0 = dq.zeros(na)

alpha2 = 2.0                 # smaller cat -> faster/clearer leak for visualization
alpha0 = jnp.sqrt(alpha2)
k2 = 4.0
k1 = 3.0                    # deliberately larger than the rate-extraction run, for visibility

L = [jnp.sqrt(k2) * (a @ a - alpha2 * dq.eye(na)),
     jnp.sqrt(k1) * a]

psi0 = dq.coherent(na, alpha0)     # start purely in the "+alpha0" branch
T = 20.0
tsave = jnp.linspace(0.0, T, 150)

res = dq.mesolve(H0, L, psi0, tsave)

# ---------------------------------------------------------------- gif -----
gif = dq.plot.wigner_gif(res.states, fps=20, xmax=4, ymax=4)
display(gif)

# --------------------------------------------------- kymograph (space-time) --
# take a 1D cut through both lobes (they sit on the real axis, y=0)
xvec, yvec, W = dq.wigner(res.states, xmax=4, ymax=4, npixels=201)
W = np.array(W)                       # shape (T, npixels, npixels)
xvec = np.array(xvec)

y0_idx = W.shape[1] // 2              # index closest to y=0
cut = W[:, : , y0_idx ]                 # shape (T, npixels): W(x, y=0, t)

plt.figure(figsize=(7,5))
extent = [xvec[0], xvec[-1], tsave[-1], tsave[0]]
plt.imshow(cut, aspect='auto', extent=extent, cmap='RdBu_r',
           vmin=-np.abs(cut).max(), vmax=np.abs(cut).max())
plt.colorbar(label="W(x, y=0, t)")
plt.axvline(float(alpha0), color='k', ls=':', lw=0.8)
plt.axvline(-float(alpha0), color='k', ls=':', lw=0.8)
plt.xlabel("x = Re(α)")
plt.ylabel("time")
plt.title(f"Wigner cut along y=0, |α0|²={alpha2}, κ1={k1}, κ2={k2}")
plt.show()

### Reading the result

- **The transfer looks smooth** — but a real bit-flip is a discrete quantum jump. `mesolve`
  integrates the ensemble-averaged master equation, which averages over *all possible jump
  times*, smearing a sharp stochastic event into a gradual fade.
- To see the discreteness, run a **single quantum trajectory** instead (`dq.jssesolve` / a
  Monte-Carlo unravelling). You should then see the weight jump abruptly from one stripe to the
  other at a random time. Comparing the two side by side shows precisely what ensemble
  averaging hides — a worthwhile exercise and the most visceral "yes, it's really protected"
  demonstration, since you can literally count the rare events.

---
# Part 8 — Multi-legged cats

## 8.1 Generalising the dissipator

Replace 2-photon with $N$-photon dissipation:

$$\mathbf{L} = \sqrt{\kappa_N}\left(\mathbf{a}^N - \alpha_0^N\right).$$

The dark states are all coherent states $|z\rangle$ with $z^N = \alpha_0^N$ — the $N$-th roots
of unity times $\alpha_0$:

$$z_k = \alpha_0\, e^{2\pi i k/N}, \qquad k = 0,\dots,N-1.$$

So $N=4$ gives **four legs** at $\alpha_0,\ i\alpha_0,\ -\alpha_0,\ -i\alpha_0$ — a
4-dimensional code space instead of 2-dimensional. The generalised mean-field drift is

$$\frac{d\alpha}{dt} = -\frac{N\kappa_N}{2}\,\bar\alpha^{\,N-1}\left(\alpha^N - \alpha_0^N\right).$$

**Why bother with 4 legs?** More structure: a 4-legged cat can detect single-photon loss as a
*syndrome* (the loss rotates you between four parity-like sectors cyclically, $\bmod\,4$
instead of $\bmod\,2$), which enables correcting rather than merely tolerating it.

**The cost, visible in the flow plot below:** at fixed $\alpha_0$, four legs sit only $90°$
apart instead of $180°$. The basins are narrower and the separatrices closer, so the
geometric protection per leg is weaker. Compare the two streamplots carefully — that
narrowing is the trade-off.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_flow(N, kappaN, alpha0, lim=3, n=25):
    x = np.linspace(-lim, lim, n); y = np.linspace(-lim, lim, n)
    X, Y = np.meshgrid(x, y)
    A = X + 1j*Y
    dA = -(N*kappaN/2) * np.conj(A)**(N-1) * (A**N - alpha0**N)
    U, V = np.real(dA), np.imag(dA)
    legs = alpha0 * np.exp(1j*2*np.pi*np.arange(N)/N)
    plt.figure(figsize=(5,5))
    plt.streamplot(X, Y, U, V, color=np.hypot(U,V), cmap='viridis', density=1.3)
    plt.scatter(legs.real, legs.imag, c='red', marker='x', s=80)
    plt.gca().set_aspect('equal'); plt.title(f"{N}-photon dissipation, {N} legs")
    plt.xlabel("Re(α)"); plt.ylabel("Im(α)"); plt.show()

plot_flow(N=2, kappaN=1.0, alpha0=2.0)   # your familiar 2-leg case, for comparison
plot_flow(N=4, kappaN=1.0, alpha0=2.0)   # the new 4-leg case

## 8.2 The 4-legged cat, adiabatic model

$$\mathbf{L} = \sqrt{\kappa_4}\left(\mathbf{a}^4 - \alpha_0^4\right), \qquad \alpha_0 = 3 .$$

**What to watch:** starting from vacuum — which is rotationally symmetric — the state fans out
and settles into **four lobes in a square**, with a rich interference pattern in the middle
(more fringe structure than the 2-legged case, since there are now $\binom{4}{2}=6$ pairs of
lobes interfering).

> ⚠️ **Numerical notes for this cell.**
> - `tsave` only runs to $t=0.05$. That is not a typo: the dissipator's norm scales like
>   $\kappa_4|\alpha_0|^4 = 81$, so the dynamics are ~80× faster than the 2-photon case. The
>   equation is correspondingly **stiff**, which is why `max_steps=1e6` is needed.
> - $\alpha_0=3$ means $|\alpha_0|^2=9$ photons *per leg*, with a 4-lobe ring of radius 3.
>   With `na=32` the Fock truncation is getting tight — **check that $\langle n\rangle$
>   saturates well below 32**, e.g. add `exp_ops=[dq.number(32)]` and inspect. If it creeps
>   toward the cutoff, the simulation is silently wrong. Raising `na` to 48–64 or lowering
>   $\alpha_0$ to 2 are both safe fixes.

In [ ]:
alpha0 = 3.0
k4 = 1.0
na = 32

a = dq.destroy(na)
H = dq.zeros(na)
L = [jnp.sqrt(k4) * (a @ a @ a @ a - alpha0**4 * dq.eye(na))]

psi0 = dq.fock(na, 0)
tsave = jnp.linspace(0, 0.05, 100)
result = dq.mesolve(H, L, psi0, tsave, method=dq.method.Tsit5(max_steps = int(1e6)))

gif = dq.plot.wigner_gif(result.states, fps=30, xmax=6, ymax=6)
display(gif)

## 8.3 The 4-legged cat with a buffer (non-adiabatic)

Exactly the same architecture as §1.3, one order higher:

$$\mathbf{H} = g_4\left(\mathbf{a}^{\dagger 4}\mathbf{b} + \mathbf{a}^4\mathbf{b}^\dagger\right)
+ \epsilon_4\left(\mathbf{b}^\dagger + \mathbf{b}\right), \qquad
\mathbf{L} = \sqrt{\kappa_b}\,\mathbf{b},$$

$$\kappa_4 = \frac{4g_4^2}{\kappa_b}, \qquad \epsilon_4 = -g_4\,\alpha_0^4 .$$

**The generalised validity condition is much harsher.** Linearising the coupling around a leg
gives an effective rate $\sim N g_N |\alpha_0|^{N-1}$, so

$$\kappa_b \;\gg\; N\,g_N\,|\alpha_0|^{N-1}.$$

For $N=4,\ \alpha_0=3$ that is $|\alpha_0|^3 = 27$ rather than $|\alpha_0|^1 = 3$ — nearly an
order of magnitude more buffer loss needed for the same quality of elimination. **Higher-order
cats are progressively harder to adiabatically eliminate**, quite apart from being harder to
engineer.

**On the hardware side:** a single buffer is sufficient (this is not a two-buffer scheme). The
difficulty is that a 4-to-1 photon exchange is a much higher-order nonlinear process than the
three-wave mixing used for $N=2$. This has been demonstrated experimentally (Campagne-Ibarcq
group, *Dissipating quartets of excitations in a superconducting circuit*), though residual
circuit nonlinearities have so far prevented stabilising a full four-legged cat.

In [ ]:
alpha0 = 3.0
alpha0_N = alpha0**4     # target for a^4 -> alpha0^4, i.e. = 16

k4 = 1.0
na, nb = 32, 8
tsave = jnp.linspace(0, 0.1, 100)

g4 = 10.0
kb = 4 * g4**2 / k4      # = 100
e4 = -g4 * alpha0_N      # -> a^4 -> -e4/g4 = alpha0_N

a = dq.destroy(na) & dq.eye(nb)
b = dq.eye(na) & dq.destroy(nb)

a4 = a @ a @ a @ a
H = g4 * (a4.dag() @ b + a4 @ b.dag()) + e4 * (b.dag() + b)
L = [jnp.sqrt(kb) * b]

psi0 = dq.fock(na, 0) & dq.fock(nb, 0)
res_full = dq.mesolve(H, L, psi0, tsave, method=dq.method.Tsit5(max_steps = int(1e6)))
rho_a = dq.ptrace(res_full.states, 0)

gif = dq.plot.wigner_gif(rho_a, fps=30, xmax=6, ymax=6)
display(gif)

## 8.4 Does the buffer model reproduce the reduced one?

The same validation as §1.4, now for $N=4$: compare fidelity to the 4-leg superposition

$$|\psi_{\text{target}}\rangle \propto \sum_{k=0}^{3} \big|\alpha_0 e^{2\pi i k/4}\big\rangle .$$

**What to watch:** if `F_full` lags or oscillates relative to `F_ad`, $\kappa_b = 4g_4^2 = 400$
is not yet deep enough into the harsher $N=4$ bad-cavity regime ($4g_4|\alpha_0|^3 = 1080$ —
note this is actually **larger** than $\kappa_b$, so some deviation is expected here!). Push
$g_4$ and $\kappa_b$ up together at fixed $\kappa_4$ and watch the agreement improve — the same
diagnostic loop you ran in §1.4.

In [ ]:
def leg_superposition(n, alpha0, N=4):
    legs = alpha0 * jnp.exp(1j * 2*jnp.pi*jnp.arange(N)/N)
    c = sum(dq.coherent(n, z) for z in legs)
    return dq.unit(c)

target = leg_superposition(na, alpha0, N=4)

na1 = dq.destroy(na)
L_ad = [jnp.sqrt(k4) * (na1@na1@na1@na1 - alpha0_N * dq.eye(na))]
res_ad = dq.mesolve(dq.zeros(na), L_ad, dq.fock(na, 0), tsave, method=dq.method.Tsit5(max_steps = int(1e6)))

F_ad   = jnp.real(dq.expect(target @ target.dag(), res_ad.states))
F_full = jnp.real(dq.expect(target @ target.dag(), rho_a))

import matplotlib.pyplot as plt
plt.plot(tsave, F_ad, label="adiabatic")
plt.plot(tsave, F_full, '--', label="full (buffer)")
plt.legend(); plt.xlabel("time"); plt.ylabel("fidelity to 4-leg superposition")

---
# Part 9 — GKP grid states

The most sophisticated bosonic code, and a genuine change of character from everything above.

## The idea

Instead of encoding in a few localised blobs, encode in a **periodic lattice** in phase space.
The code is defined by two commuting **stabilisers**, which are displacement operators:

$$\mathbf{S}_q = e^{i\eta\mathbf{q}}, \qquad \mathbf{S}_p = e^{i\eta\mathbf{p}},$$

with lattice constant $\eta = \sqrt{2\pi d}$ for a qudit of dimension $d$; a qubit uses
$\eta_\square = 2\sqrt\pi$, and the simplified scheme below uses $\eta = \eta_\square/2 = \sqrt\pi$.

Ideal GKP states are infinite Dirac combs — unnormalisable, infinite energy. Physical
("finite-energy") GKP states apply a Gaussian envelope
$\mathbf{E}_\epsilon = e^{-\frac{\epsilon}{2}(\mathbf{q}^2+\mathbf{p}^2)}$, with
$\langle n\rangle \propto 1/\epsilon$.

## The stabilising dissipators

This notebook uses the recent **two-dissipator** scheme (Robin, Rouchon & Sellem 2026 — who,
pleasingly, ran their own simulations in `dynamiqs`):

$$\mathbf{M}_1 = \sin(\eta\mathbf{q}) + i\epsilon\cos(\eta\mathbf{q})\,\mathbf{p},$$
$$\mathbf{M}_2 = \sin(\eta\mathbf{p}) - i\epsilon\cos(\eta\mathbf{p})\,\mathbf{q}.$$

**Term-by-term — note the clean division of labour:**

| term | role |
|---|---|
| $\sin(\eta\mathbf{q})$ | vanishes when $\mathbf{q} \equiv 0 \pmod{\pi/\eta}$ — this is the term that **builds the grid** in $q$ |
| $i\epsilon\cos(\eta\mathbf{q})\mathbf{p}$ | the **envelope / energy-trimming** term; it makes the state finite-energy and supplies the actual contraction, at rate $\lambda \approx 2\epsilon\eta$ |
| $\mathbf{M}_2$ | the same pair of jobs in the conjugate quadrature |

Without the $\epsilon$ terms you would stabilise an infinite-energy comb (not normalisable);
without the $\sin$ terms you would have no lattice. **Both are required, and $\epsilon$ sets
both the envelope size and the speed.**

## Implementation trick

You never need a matrix exponential: the stabilisers *are* displacement operators,

$$e^{i\eta\mathbf{q}} = \mathbf{D}\!\left(\tfrac{i\eta}{\sqrt2}\right), \qquad
e^{i\eta\mathbf{p}} = \mathbf{D}\!\left(-\tfrac{\eta}{\sqrt2}\right),$$

so $\sin$ and $\cos$ follow from $(\mathbf{U}\pm\mathbf{U}^\dagger)$.

## Two structural contrasts with cats

1. **GKP genuinely needs two dissipation channels** — one per quadrature. Unlike the 4-legged
   cat (which needed one higher-order channel), you cannot collapse these into one.
2. **Truncation matters much more.** Grid states occupy a wide region of phase space, hence
   many Fock levels. Hence `n = 128` here rather than 32. Always check $\langle n\rangle$
   saturates well below the cutoff.

Starting from vacuum with $\eta=\sqrt\pi$, the steady state is close to the **magic state**
$\cos(\pi/8)|{+}Z\rangle + \sin(\pi/8)|{-}Z\rangle$ (fidelity $>92\%$ at $\epsilon=0.15$).
Using $\eta=\sqrt{\pi/2}$ instead gives a unique steady state — the **qunaught**, of interest
for metrology.

**What to watch:** a single vacuum blob gradually organising into a **square array of
peaks**, with negative regions in between, all under a Gaussian envelope. Convergence rate
$\lambda \approx 2\epsilon\eta \approx 0.53$, hence $t \sim 20$.

In [ ]:
import jax.numpy as jnp
import dynamiqs as dq

n = 128                      # Hilbert space cutoff — check <n> saturates well below this
eta = jnp.sqrt(jnp.pi)      # qubit; use jnp.sqrt(jnp.pi/2) for a qunaught
eps = 0.15

a = dq.destroy(n)
q = (a + a.dag()) / jnp.sqrt(2)
p = -1j * (a - a.dag()) / jnp.sqrt(2)

Uq = dq.displace(n, 1j * eta / jnp.sqrt(2))   # exp(i eta q)
Up = dq.displace(n, -eta / jnp.sqrt(2))       # exp(i eta p)

cos_eq = (Uq + Uq.dag()) / 2
sin_eq = (Uq - Uq.dag()) / (2j)
cos_ep = (Up + Up.dag()) / 2
sin_ep = (Up - Up.dag()) / (2j)

M1 = sin_eq + 1j * eps * cos_eq @ p
M2 = sin_ep - 1j * eps * cos_ep @ q

H = dq.zeros(n)
L = [M1, M2]

psi0 = dq.fock(n, 0)                  # vacuum -> converges toward a GKP magic state
tsave = jnp.linspace(0, 20.0, 200)    # rate ~ 2*eps*eta ~ 0.5, so t~20 is comfortably converged

result = dq.mesolve(H, L, psi0, tsave, method=dq.method.Tsit5(max_steps = int(1e6)))

gif = dq.plot.wigner_gif(result.states, fps=30, xmax=5, ymax=5)
display(gif)

### 9.1 GKP with buffers (non-adiabatic)

The natural extension of the buffer architecture, with **one buffer per dissipator**:

$$\mathbf{H} = g\left(\mathbf{M}_1\mathbf{b}_1^\dagger + \mathbf{M}_1^\dagger\mathbf{b}_1\right)
+ g\left(\mathbf{M}_2\mathbf{b}_2^\dagger + \mathbf{M}_2^\dagger\mathbf{b}_2\right),
\qquad \mathbf{L} = \left[\sqrt{\kappa_b}\,\mathbf{b}_1,\ \sqrt{\kappa_b}\,\mathbf{b}_2\right],$$

with the same $\kappa_{\text{eff}} \approx 4g^2/\kappa_b$ logic as everywhere else.

> ⚠️ **Honest caveat.** Unlike the cat sections — where the buffer circuit corresponds to a
> published, experimentally realised design — this two-buffer GKP construction is the natural
> *theoretical extrapolation* of the pattern, not a validated hardware blueprint. Real GKP
> experiments to date (Yale, Google, trapped ions) overwhelmingly use **discrete-time,
> ancilla-qubit-mediated** protocols (sharpen–trim, big–small–big) rather than continuous
> engineered dissipation, precisely because couplings of the form $\sin(\eta\mathbf{q})$ are far
> harder to engineer than the polynomial couplings that cats need. Treat §9.1 as a
> pattern-completion exercise; §9 above is the one backed by the literature.

> ⚠️ **Cost warning.** The Hilbert space here is $40 \times 6 \times 6 = 1440$, so the density
> matrix is $1440^2 \approx 2\times10^6$ complex numbers. This cell is *much* slower than
> everything above and the memory cutoff `n=40` is tighter than the `n=128` used for the
> single-mode version — expect visible truncation artefacts in the grid. Reduce `tsave`
> resolution first if you need it to finish.

In [ ]:
n, nb = 40, 6
tsave = jnp.linspace(0, 20.0, 200)

g, kb = 2.0, 20.0     # keep g << kb, same bad-cavity logic as before

a  = dq.destroy(n) & dq.eye(nb) & dq.eye(nb)
b1 = dq.eye(n) & dq.destroy(nb) & dq.eye(nb)
b2 = dq.eye(n) & dq.eye(nb) & dq.destroy(nb)

q = (a + a.dag()) / jnp.sqrt(2)
p = -1j * (a - a.dag()) / jnp.sqrt(2)
Uq = dq.displace(n, 1j*eta/jnp.sqrt(2)) & dq.eye(nb) & dq.eye(nb)
Up = dq.displace(n, -eta/jnp.sqrt(2)) & dq.eye(nb) & dq.eye(nb)
cos_eq = (Uq + Uq.dag())/2; sin_eq = (Uq - Uq.dag())/(2j)
cos_ep = (Up + Up.dag())/2; sin_ep = (Up - Up.dag())/(2j)
M1 = sin_eq + 1j*eps*cos_eq@p
M2 = sin_ep - 1j*eps*cos_ep@q

H = g*(M1 @ b1.dag() + M1.dag() @ b1) + g*(M2 @ b2.dag() + M2.dag() @ b2)
L = [jnp.sqrt(kb)*b1, jnp.sqrt(kb)*b2]

psi0 = dq.fock(n,0) & dq.fock(nb,0) & dq.fock(nb,0)
res_full = dq.mesolve(H, L, psi0, tsave, method=dq.method.Tsit5(max_steps = int(1e6)))
rho_gkp = dq.ptrace(res_full.states, 0)
gif = dq.plot.wigner_gif(rho_gkp, fps=30, xmax=5, ymax=5)
display(gif)

---
# Part 10 — Summary: what every term does

## The operator dictionary

| operator / term | appears in | moves what | effect on logical information |
|---|---|---|---|
| $\sqrt{\kappa_2}(\mathbf{a}^2-\alpha_0^2)$ | dissipative cat | population toward $\pm\alpha_0$ | **stabilises**: confines to the code space, suppresses bit-flips exponentially. Conserves parity, so it cannot fix phase-flips |
| $g_2(\mathbf{a}^{\dagger2}\mathbf{b}+\text{h.c.})$ | buffer circuit | 2 memory photons $\leftrightarrow$ 1 buffer photon | **transports** entropy out of the memory; reversible alone |
| $\epsilon_2(\mathbf{b}^\dagger+\mathbf{b})$ | buffer circuit | — | **selects** the target: $\alpha_0^2=-\epsilon_2/g_2$ |
| $\sqrt{\kappa_b}\,\mathbf{b}$ | buffer circuit | buffer $\to$ environment | **the only entropy sink**. No $\kappa_b$, no stabilisation |
| $\sqrt{\kappa_1}\,\mathbf{a}$ | everywhere (noise) | flips parity exactly, $\mathbf{a}\|\mathcal{C}^+\rangle\propto\|\mathcal{C}^-\rangle$ | **corrupts phase** (logical $Z$). Rate $\propto\kappa_1\|\alpha\|^2$. With stabilisation off, also drains energy |
| $-K\mathbf{a}^{\dagger2}\mathbf{a}^2$ | Kerr cat | rotates/shears phase space | **confines** via an energy gap; generates negativity; **no attractor** |
| $\epsilon_2(\mathbf{a}^{\dagger2}+\mathbf{a}^2)$ | Kerr cat | — | picks the axis, sets $\alpha_0^2=\epsilon_2/K$ |
| $\sqrt{\kappa_4}(\mathbf{a}^4-\alpha_0^4)$ | 4-legged cat | population toward 4 legs | **stabilises** a 4-dim code; enables loss *detection*, at the cost of narrower basins |
| $\sin(\eta\mathbf{q})$ | GKP | — | **builds the lattice** in $q$ |
| $i\epsilon\cos(\eta\mathbf{q})\mathbf{p}$ | GKP | shrinks the envelope | makes it **finite-energy**; sets the convergence rate $\approx 2\epsilon\eta$ |

## The three stabilisation philosophies

| | mechanism | code space is | attracts arbitrary states? | needs extra ingredient? |
|---|---|---|---|---|
| **dissipative cat** | entropy removal | $\ker \mathbf{L}$ | **yes** | a lossy buffer |
| **Kerr cat** | energy gap | ground manifold of $\mathbf{H}$ | **no** | $\kappa_1$ to relax into it |
| **GKP** | entropy removal | joint stabiliser $+1$ eigenspace | **yes** | two channels; hard couplings |

## The five conclusions worth carrying away

1. **Dark states are the design principle.** Every scheme is "choose $\mathbf{L}$ so that
   $\mathbf{L}|\psi_{\text{code}}\rangle=0$". Everything else is engineering.
2. **Rate and target are independent knobs.** $\kappa_2$ sets speed; the offset ($\alpha_0^2$,
   or $\epsilon/g$) sets the destination. Confusing them causes most parameter bugs.
3. **What matters is always a ratio, never a single coupling.** $\kappa_2 = 4g_2^2/\kappa_b$
   and the validity condition $\kappa_b \gg 2g_2|\alpha_0|$ mean $(g_2,\kappa_b)$ must scale
   **together**. Strong coupling demands a *more* lossy buffer, not less.
4. **The noise bias is the payoff.** Bit-flips $\sim e^{-c|\alpha|^2}$, phase-flips
   $\sim\kappa_1|\alpha|^2$. Linear cost, exponential gain — you buy a classically-correctable
   error model.
5. **Stabilisation corrects leakage, not logical errors.** Part 5's re-stabilisation restores
   $\langle n\rangle$ and the manifold populations, but never parity or purity. Autonomous
   stabilisation gets you a *biased-noise qubit*, not an error-corrected one — the remaining
   phase-flips still need an outer code.

## Suggested next experiments

- **Quantum trajectories** (`dq.jssesolve`): watch individual discrete bit-flip and
  phase-flip jumps instead of smooth ensemble averages. The single most illuminating follow-up.
- **Kerr-cat energy gap**: diagonalise $\mathbf{H}$ directly (`np.linalg.eigvalsh`) and plot the
  ground-doublet splitting vs $|\alpha|^2$ on a log axis — the Hamiltonian analogue of Part 6,
  with no time evolution needed.
- **Squeezed cats**: add a squeezing drive and reproduce the reported large bit-flip-time
  improvements.
- **Gates**: apply a slow displacement $\epsilon_Z(\mathbf{a}+\mathbf{a}^\dagger)$ while
  stabilised and watch a logical $Z$ rotation happen *without* leaving the manifold.
- **Convergence-rate scaling**: extract $\lambda$ from Part 5's phase-3 recovery and check it
  against $\kappa_2|\alpha_0|^2$.